# Performance

What CrocoDash's expensive steps cost on NCAR's machines, with fits for sizing jobs. The numbers
come from the workflow ([`test_workflow.py`](https://github.com/CROCODILE-CESM/CrocoDashChecks/blob/main/performance/test_workflow.py)): every grid in the
[test matrix](index.ipynb) is built once, with its bathymetry, and each forcing turned on for it
runs on it; then a few sweeps vary what isn't a grid (source resolution, days, workers, smoothing
radius) on one or two grids. Each section says what was measured, gives the fit and what it
means, and plots the measurements (dots) against the fit. Every grid's own case is in a table
with its result; the sweeps are folded under "Show the numbers".

The workflow runs by hand (see [How it works](how_it_works.md)); this page refits its results
every time the site is built.

In [ ]:
%config InlineBackend.figure_formats = ["svg"]
import math

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter, NullFormatter

import checks
from checks import fmt_gb, fmt_s

plt.rcParams.update(
    {
        "font.size": 9,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
wf = checks.workflow()
grids = checks.matrix()


def plain_numbers(fig):
    """Label log axes 1, 10, 100, 1K, 1M, 1B rather than 10⁰, 10¹, 10²."""

    def label(v, _):
        for size, unit in ((1e9, "B"), (1e6, "M"), (1e3, "K")):
            if v >= size:
                return f"{v / size:g}{unit}"
        return f"{v:g}"

    for a in fig.axes:
        for axis, scale in ((a.xaxis, a.get_xscale()), (a.yaxis, a.get_yscale())):
            if scale == "log":
                axis.set_major_formatter(FuncFormatter(label))
                axis.set_minor_formatter(NullFormatter())


def fit(terms, measured):
    """Least-squares coefficients c with measured ≈ terms @ c (one row of terms per case)."""
    return np.linalg.lstsq(
        np.array(terms, float), np.array(measured, float), rcond=None
    )[0]


def fit_error(predicted, measured):
    """How far a fit is from the measurements, e.g. 'median 5%, worst 20%'."""
    error = np.abs(np.array(predicted) - np.array(measured)) / np.array(measured)
    return f"median {np.median(error):.0%}, worst {error.max():.0%}"


def cases(test):
    """The cases of one test that passed, e.g. test_bc."""
    return [b for name, b in wf["cases"].items() if name.startswith(test + "[")]


def info(b, key):
    """A value the workflow recorded about its case (its extra_info)."""
    return b["extra_info"][key]


def seconds(b):
    return b["stats"]["mean"]


def peak_gb(b):
    return info(b, "peak_mb") / 1024


def rows_with(forcing):
    """The matrix rows a forcing runs on."""
    return [name for name, grid in grids.items() if forcing in grid.get("forcings", [])]


def grid_table(test, rows, headers, cells):
    """A line per matrix row: its name, then cells(its case), or its result if it has none."""
    lines = []
    for name in rows:
        test_id = f"{test}[{name}]"
        case = wf["cases"].get(test_id)
        if case:
            lines.append([f"`{name}`", *cells(case)])
        else:
            _, why = wf["outcomes"].get(test_id, (None, ""))
            result = checks.step(wf, test_id) + (
                f": {checks.plain(why)}" if why else ""
            )
            lines.append([f"`{name}`", result] + ["—"] * (len(headers) - 2))
    return checks.table(headers, lines)

(bathymetry)=
## Grids and bathymetry

Building each grid with its bathymetry and vertical grid, from its case config with CrocoDash's
own builders, as `crocodash create` does, and writing the files a case gets. Nearly all of the
time is mom6_forge's `Topo.set_from_dataset`: GEBCO interpolated onto the grid. The fit is over the
`pacific_*` squares from 2.5° to 40° at 0.1°, which use full-resolution GEBCO (15″); the other
grids use GEBCO coarsened 8× (1/30°), like the CESM runs, because full GEBCO over a polar cap or
a bigger box needs more memory than a node has.

In [ ]:
FULL_GEBCO = "GEBCO_2024.nc"
GEBCO_NAMES = {FULL_GEBCO: "GEBCO 15″", "GEBCO_2024_coarse_x8.nc": "GEBCO 1/30°"}
built = cases("test_grid")
full = sorted(
    (b for b in built if info(b, "bathymetry") == FULL_GEBCO and info(b, "area_deg2")),
    key=lambda b: info(b, "area_deg2"),
)
area = np.array([info(b, "area_deg2") for b in full])  # square degrees
grid_s = np.array([seconds(b) for b in full])
grid_gb = np.array([peak_gb(b) for b in full])


def power_fit(x, y):
    """(a, p) with y ≈ a·x^p, fitted as a straight line in log-log space."""
    p, log_a = np.polyfit(np.log(x), np.log(y), 1)
    return np.exp(log_a), p


(ta, tp), (ma, mp) = power_fit(area, grid_s), power_fit(area, grid_gb)
checks.show(f"""
**The fit**, over {len(full)} grids built from full GEBCO (A is the grid's area in square degrees):

- time (s) ≈ {ta:.2g} · A{{sup}}`{tp:.2f}`
- peak memory (GB) ≈ {ma:.2g} · A{{sup}}`{mp:.2f}`

Time {fit_error(ta * area**tp, grid_s)}; memory {fit_error(ma * area**mp, grid_gb)}.

- **Time** grows with area: {fmt_s(ta * 1600**tp)} for a 40° square.
- **Memory** too: {fmt_gb(ma * 1600**mp)} for a 40° square.
""")

coarse = [
    b for b in built if info(b, "bathymetry") != FULL_GEBCO and info(b, "area_deg2")
]
fig, ax = plt.subplots(1, 2, figsize=(8, 3.6), layout="constrained")
span = np.geomspace(area[0], area[-1], 40)
for a, measured, (c, p), what in (
    (ax[0], seconds, (ta, tp), "time (s)"),
    (ax[1], peak_gb, (ma, mp), "peak memory (GB)"),
):
    (line,) = a.loglog(span, c * span**p, "-", label="GEBCO 15″: fit")
    a.loglog(
        area,
        [measured(b) for b in full],
        "o",
        color=line.get_color(),
        label="GEBCO 15″",
    )
    a.loglog(
        [info(b, "area_deg2") for b in coarse],
        [measured(b) for b in coarse],
        "s",
        label="GEBCO 1/30° (no fit)",
    )
    a.set(
        title=f"{what.split(' (')[0].capitalize()} by grid area",
        xlabel="square degrees (A)",
        ylabel=what,
    )
ax[0].legend()
plain_numbers(fig)
plt.show()

In [ ]:
def grid_fit(b, c, p):
    a = info(b, "area_deg2")
    return c * a ** p if info(b, "bathymetry") == FULL_GEBCO and a else None


checks.show(
    grid_table(
        "test_grid",
        list(grids),
        [
            "Grid",
            "Bathymetry",
            "Domain cells",
            "Ocean cells",
            "Time",
            "Fit",
            "Peak memory",
            "Fit",
        ],
        lambda b: [
            GEBCO_NAMES.get(info(b, "bathymetry"), info(b, "bathymetry")),
            f"{info(b, 'n_cells'):,}",
            f"{info(b, 'ocean_cells'):,}",
            fmt_s(seconds(b)),
            fmt_s(grid_fit(b, ta, tp)) if grid_fit(b, ta, tp) else "—",
            fmt_gb(peak_gb(b)),
            fmt_gb(grid_fit(b, ma, mp)) if grid_fit(b, ma, mp) else "—",
        ],
    )
)

(obc)=
## Open boundaries

CrocoDash's open-boundary generation (`ConditionsConfigurator.process_bc`): regridding the
source ocean onto each boundary of the domain, then merging the pieces. Each grid with open
boundaries turned on runs one case: a 0.25° source, 30 days, 1 worker. The sweeps vary the
source resolution and the days on `pacific_10deg`, and the workers over 360 days there and on
`pacific_80deg`.

:::{note}
The source is CrocoDash's synthetic `reference_ocean`, so nothing is downloaded and the download
isn't timed. It has 10 levels; GLORYS has 50.
:::

In [ ]:
SWEEP_ROW = "pacific_10deg"  # test_workflow.py's BC_SWEEP_ROW
bc = cases("test_bc") + cases("test_bc_sweep")


def obc_terms(e):
    """One case's terms in the time fit and in the memory fit.

    REGRID splits the days into 30-day slices and hands min(workers, slices) forked
    workers an equal share each; a single worker runs in the main process."""
    S = e["slices"]  # 30-day slices
    k = math.ceil(S / e["workers"])  # slices per worker
    w = min(e["workers"], S) if e["workers"] > 1 else 0  # forked workers
    P = e["n_src"] / 1e6  # million source points
    C = e["n_cells"] / 1e6  # million domain cells
    time_terms = [1, S, S * C, k, k * P, k * C]
    memory_terms = [1, P, C, w, w * C]
    return time_terms, memory_terms


t = fit([obc_terms(b["extra_info"])[0] for b in bc], [seconds(b) for b in bc])
m = fit([obc_terms(b["extra_info"])[1] for b in bc], [peak_gb(b) for b in bc])


def obc_fit(e):
    """(seconds, GB) the fits give for a case."""
    time_terms, memory_terms = obc_terms(e)
    return float(np.dot(t, time_terms)), float(np.dot(m, memory_terms))


fit_s = lambda b: obc_fit(b["extra_info"])[0]
fit_gb = lambda b: obc_fit(b["extra_info"])[1]
checks.show(f"""
**The fit**, over all {len(bc)} cases:

- time (s) ≈ {t[0]:.2g} + S × ({t[1]:.2g} + {t[2]:.2g} C) + k × ({t[3]:.2g} + {t[4]:.2g} P + {t[5]:.2g} C)
- memory (GB) ≈ {m[0]:.2g} + {m[1]:.2g} P + {m[2]:.2g} C + w × ({m[3]:.2g} + {m[4]:.2g} C)

S is the number of 30-day slices, k slices per worker, w workers (0 for one), P million source
points and C million domain cells. Time {fit_error([fit_s(b) for b in bc], [seconds(b) for b in bc])};
memory {fit_error([fit_gb(b) for b in bc], [peak_gb(b) for b in bc])}.
""")

In [ ]:
def on_sweep_row(b, **fixed):
    return info(b, "row") == SWEEP_ROW and all(
        info(b, k) == v for k, v in fixed.items()
    )


by_grid = sorted(cases("test_bc"), key=lambda b: info(b, "n_cells"))
by_source = sorted(
    (b for b in bc if on_sweep_row(b, days=30, workers=1)),
    key=lambda b: info(b, "n_src"),
)
by_days = sorted(
    (b for b in bc if on_sweep_row(b, source_res_deg=0.25, workers=1)),
    key=lambda b: info(b, "days"),
)
by_workers = {}  # (grid, source resolution) -> its 360-day cases, by worker count
for b in sorted(
    (b for b in bc if info(b, "days") == 360), key=lambda b: info(b, "workers")
):
    by_workers.setdefault((info(b, "row"), info(b, "source_res_deg")), []).append(b)

d0 = by_days[0]["extra_info"]
per_slice = obc_fit({**d0, "days": 60, "slices": 2})[0] - obc_fit(d0)[0]
speedups = [seconds(rows[0]) / seconds(rows[-1]) for rows in by_workers.values()]
most = max(info(b, "workers") for rows in by_workers.values() for b in rows)
checks.show(f"""
- **Source points:** linear, {t[4]:.0f} s more per million source points per 30 days (`{SWEEP_ROW}`, 1 worker).
- **Domain cells:** linear too, {t[2] + t[5]:.0f} s more per million domain cells per 30 days (0.25° source, 1 worker).
- **Days:** linear, {per_slice:.1f} s per 30 days (`{SWEEP_ROW}`, 1 worker).
- **Workers:** time falls as 1/workers toward a floor: {min(speedups):.1f}–{max(speedups):.1f}× faster on
  {most} workers than on 1. Each worker takes whole slices, so with 12 slices, 6 to 11 workers all
  take as long as 6: use a worker count that divides the slices evenly.
- **Memory:** {m[2]:.1f} GB per million domain cells, plus {m[3]:.2f} GB per worker and {m[4]:.2f} GB
  per worker per million domain cells.
""")

fig, ax = plt.subplots(2, 3, figsize=(12, 7), layout="constrained")


def plot(ax, rows, x, y, fit_y, line=True):
    """Measured dots, and the fit through the same cases: a line, or crosses
    when the cases differ in more than x."""
    xs = [x(b) for b in rows]
    (fitted,) = ax.plot(xs, [fit_y(b) for b in rows], "-" if line else "x", label="fit")
    ax.plot(xs, [y(b) for b in rows], "o", color=fitted.get_color(), label="measured")


plot(ax[0, 0], by_source, lambda b: info(b, "n_src") / 1e6, seconds, fit_s)
ax[0, 0].set(
    title="Time by source points", xlabel="million source points (P)", ylabel="time (s)"
)
plot(ax[0, 1], by_grid, lambda b: info(b, "n_cells") / 1e6, seconds, fit_s, line=False)
ax[0, 1].set(title="Time by grid", xlabel="million domain cells (C)", ylabel="time (s)")
ax[0, 1].legend()
plot(ax[0, 2], by_days, lambda b: info(b, "days"), seconds, fit_s)
ax[0, 2].set(title="Time by days", xlabel="days", ylabel="time (s)")
for (row, source), rows in by_workers.items():
    label = f"{row}, {source:g}° source"
    workers = range(1, info(rows[-1], "workers") + 1)
    for a, measured, which in ((ax[1, 0], seconds, 0), (ax[1, 1], peak_gb, 1)):
        (line,) = a.plot(
            workers,
            [obc_fit({**rows[0]["extra_info"], "workers": w})[which] for w in workers],
            "-",
            label=label,
        )
        a.plot(
            [info(b, "workers") for b in rows],
            [measured(b) for b in rows],
            "o",
            color=line.get_color(),
        )
ax[1, 0].set(title="Time by workers (360 days)", xlabel="workers", ylabel="time (s)")
ax[1, 1].set(
    title="Peak memory by workers (360 days)",
    xlabel="workers",
    ylabel="peak memory (GB)",
)
ax[1, 0].legend()
plot(ax[1, 2], by_grid, lambda b: info(b, "n_cells") / 1e6, peak_gb, fit_gb, line=False)
ax[1, 2].set(
    title="Peak memory by grid",
    xlabel="million domain cells (C)",
    ylabel="peak memory (GB)",
)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3.6), layout="constrained")
for a, fitted, measured, unit, what in (
    (ax[0], fit_s, seconds, "s", "time"),
    (ax[1], fit_gb, peak_gb, "GB", "peak memory"),
):
    xs = [fitted(b) for b in bc]
    ys = [measured(b) for b in bc]
    a.loglog(xs, ys, "o")
    lo, hi = min(xs + ys) / 1.5, max(xs + ys) * 1.5
    a.plot([lo, hi], [lo, hi], "-", color="grey", label="fit = measured")
    a.set(
        title=f"Fit against measurement: {what}",
        xlabel=f"fit ({unit})",
        ylabel=f"measured ({unit})",
    )
    a.legend()
plain_numbers(fig)
plt.show()

In [ ]:
OBC_HEADERS = ["Source points", "Domain cells", "Time", "Fit", "Peak memory", "Fit"]


def obc_cells(b):
    return [
        f"{info(b, 'n_src'):,}",
        f"{info(b, 'n_cells'):,}",
        fmt_s(seconds(b)),
        fmt_s(fit_s(b)),
        fmt_gb(peak_gb(b)),
        fmt_gb(fit_gb(b)),
    ]


checks.show(
    "**By grid** (0.25° source, 30 days, 1 worker): does crossing a seam or covering a pole cost "
    "more than the fit predicts?\n\n"
    + grid_table("test_bc", rows_with("bc"), ["Grid", *OBC_HEADERS], obc_cells)
)

In [ ]:
sweep = cases("test_bc_sweep")
text = ""
for title, rows in (
    ("By source resolution", by_source),
    ("By days", by_days),
    ("By workers (360 days)", [b for rows in by_workers.values() for b in rows]),
):
    text += (
        f"**{title}**\n\n"
        + checks.table(
            ["Grid", "Source", "Days", "Workers", *OBC_HEADERS],
            [
                [
                    f"`{info(b, 'row')}`",
                    f"{info(b, 'source_res_deg'):g}°",
                    info(b, "days"),
                    info(b, "workers"),
                    *obc_cells(b),
                ]
                for b in rows
            ],
        )
        + "\n\n"
    )
checks.show(text)

(runoff)=
## Runoff mapping

CrocoDash's runoff mapping (`RunoffConfigurator.process`, which calls mom6_forge's
`gen_rof_maps`): the maps that carry GLOFAS's global river runoff to the ocean grid, nearest
neighbour and then smoothed over a radius rmax. Each grid with runoff turned on runs one case at
rmax 20 km: a ladder of coasts from a 1° box to the Indo-Pacific, and the Gulf of Mexico at
several resolutions. The sweep varies rmax on three of them. The smoothed map's weights are also
what CESM's coupler holds in memory while the model runs.

In [ ]:
rof = cases("test_runoff") + cases("test_runoff_sweep")

# CESM's coupler holds the smoothed map as two route handles (liquid and ice runoff),
# ~167 bytes a weight each, measured on CrocIndoPacific_DROF_112 (4.0e8 weights,
# 133 GB on the mediator node).
COUPLER_BYTES_PER_WEIGHT = 2 * 167


def coupler_gb(weights):
    return weights * COUPLER_BYTES_PER_WEIGHT / 1024**3


def spread(e, rmax=None):
    """pi rmax^2 / a: about how many ocean cells lie within rmax of a coastal cell."""
    r = e["rmax_km"] if rmax is None else rmax
    return math.pi * r**2 / e["ocn_cell_km2"]


# Weights: n_s = N (1 + k spread), N the nearest-neighbour map's weights. An rmax
# narrower than a cell spreads nowhere (n_s = N), so those cases stay out of the fit.
wide = [b for b in rof if info(b, "rmax_km") ** 2 > info(b, "ocn_cell_km2")]
(k,) = fit(
    [[spread(b["extra_info"])] for b in wide],
    [info(b, "nnsm_n_s") / info(b, "nn_n_s") - 1 for b in wide],
)


def weights_fit(e, rmax=None):
    return e["nn_n_s"] * (1 + k * spread(e, rmax))


def cells_and_weights(b):
    """(million ocean cells, billion weights in the smoothed map)."""
    return info(b, "n_ocn") / 1e6, info(b, "nnsm_n_s") / 1e9


rof_s = [seconds(b) for b in rof]
rof_gb = [peak_gb(b) for b in rof]
rt = fit([[1, *cells_and_weights(b)] for b in rof], rof_s)
rm = fit([[1, cells_and_weights(b)[1]] for b in rof], rof_gb)
rof_fit_s = (
    lambda b: rt[0] + rt[1] * cells_and_weights(b)[0] + rt[2] * cells_and_weights(b)[1]
)
rof_fit_gb = lambda b: rm[0] + rm[1] * cells_and_weights(b)[1]

# Weights against rmax: one exponent, shared by the grids the sweep runs on.
swept = list(dict.fromkeys(info(b, "row") for b in cases("test_runoff_sweep")))
rmax_rows = sorted(
    (b for b in rof if info(b, "row") in swept), key=lambda b: info(b, "rmax_km")
)
p = fit(
    [
        [float(info(b, "row") == d) for d in swept] + [math.log(info(b, "rmax_km"))]
        for b in rmax_rows
    ],
    [math.log(info(b, "nnsm_n_s")) for b in rmax_rows],
)[-1]


def weights(n):
    return f"{n / 1e9:.2f} billion" if n >= 1e8 else f"{n / 1e6:.1f} million"


checks.show(f"""
**The fit**, over {len(rof)} cases:

- time (s) ≈ {rt[0]:.3g} + {rt[1]:.3g} C + {rt[2]:.3g} W
- memory (GB) ≈ {rm[0]:.3g} + {rm[1]:.3g} W
- weights ≈ N · (1 + {k:.2f} · π r² / a)
- coupler memory (GB) ≈ {coupler_gb(1e9):.0f} W

C is million ocean cells, W billion weights in the smoothed map, N the GLOFAS cells in the
grid's box (its area / 0.0025 deg²), r rmax (km) and a the ocean cells' area (km²). Time
{fit_error([rof_fit_s(b) for b in rof], rof_s)}; memory {fit_error([rof_fit_gb(b) for b in rof], rof_gb)};
weights {fit_error([weights_fit(b["extra_info"]) for b in wide], [info(b, "nnsm_n_s") for b in wide])}
(cases with rmax wider than a cell).

- **Time** is flat at {rt[0]:.0f} s, set by the global GLOFAS mesh, then {rt[1]:.0f} s more per
  million ocean cells.
- **Weights** go as about rmax{{sup}}`{p:.1f}`: doubling rmax multiplies the weights by {2**p:.1f}, and the
  coupler's memory with them.
- **Memory** is flat at {rm[0]:.0f} GB, set by the global GLOFAS mesh, then {rm[1]:.0f} GB more per
  billion weights.
""")

fig, ax = plt.subplots(1, 3, figsize=(12, 3.8), layout="constrained")
by_cells = sorted(cases("test_runoff"), key=lambda b: info(b, "n_ocn"))
xs = [info(b, "n_ocn") for b in by_cells]
(line,) = ax[0].semilogx(xs, [rof_fit_s(b) for b in by_cells], "x", label="fit")
ax[0].semilogx(
    xs, [seconds(b) for b in by_cells], "o", color=line.get_color(), label="measured"
)
ax[0].set(
    title="Time by ocean cells (rmax 20 km)", xlabel="ocean cells", ylabel="time (s)"
)
ax[0].legend()

r_span = np.geomspace(
    min(info(b, "rmax_km") for b in rmax_rows),
    max(info(b, "rmax_km") for b in rmax_rows),
    40,
)
for d in swept:
    rows = [b for b in rmax_rows if info(b, "row") == d]
    (line,) = ax[1].loglog(
        r_span, [weights_fit(rows[0]["extra_info"], r) for r in r_span], "-", label=d
    )
    ax[1].loglog(
        [info(b, "rmax_km") for b in rows],
        [info(b, "nnsm_n_s") for b in rows],
        "o",
        color=line.get_color(),
    )
ax[1].set(title="Weights by rmax", xlabel="rmax (km)", ylabel="smoothed-map weights")
ax[1].set_xticks([10, 20, 40, 80])
ax[1].legend()

n_all = [info(b, "nnsm_n_s") for b in rof]
n_span = np.geomspace(min(n_all), max(n_all), 40)
(line,) = ax[2].semilogx(n_span, rm[0] + rm[1] * n_span / 1e9, "-")
ax[2].semilogx(n_all, rof_gb, "o", color=line.get_color())
ax[2].set(
    title="Peak memory by weights",
    xlabel="smoothed-map weights",
    ylabel="peak memory (GB)",
)
plain_numbers(fig)
plt.show()

In [ ]:
ROF_HEADERS = [
    "Ocean cells",
    "Weights",
    "Fit",
    "Time",
    "Fit",
    "Peak memory",
    "Fit",
    "Coupler",
]


def rof_cells(b):
    return [
        f"{info(b, 'n_ocn'):,}",
        weights(info(b, "nnsm_n_s")),
        weights(weights_fit(b["extra_info"])),
        fmt_s(seconds(b)),
        fmt_s(rof_fit_s(b)),
        fmt_gb(peak_gb(b)),
        fmt_gb(rof_fit_gb(b)),
        fmt_gb(coupler_gb(info(b, "nnsm_n_s"))),
    ]


checks.show(
    "**By grid** (rmax 20 km)\n\n"
    + grid_table(
        "test_runoff",
        rows_with("runoff"),
        ["Grid", "Resolution", *ROF_HEADERS],
        lambda b: [checks.resolution(grids[info(b, "row")]["grid"]), *rof_cells(b)],
    )
)

In [ ]:
checks.show(
    "**By rmax**\n\n"
    + checks.table(
        ["Grid", "rmax", *ROF_HEADERS],
        [
            [f"`{info(b, 'row')}`", f"{info(b, 'rmax_km')} km", *rof_cells(b)]
            for b in rmax_rows
        ],
    )
)

(mom6)=
## MOM6 speed vs. NTASKS_OCN

How MOM6's run time falls as it gets more MPI tasks, on 100×100×10 grids on Derecho: a sweep over
one Bahamas domain, and four more domains (the Caribbean, the Equatorial Pacific, the Arctic and
the Antarctic). These were run by hand; the scripts are in the CROC monorepo's
`dev/scaling_test/` and the results in
[`mom6_scaling.json`](https://github.com/CROCODILE-CESM/CrocoDashChecks/blob/main/performance/mom6_scaling.json).

In [ ]:
scaling = checks.load(checks.PERF / "mom6_scaling.json")
groups = {
    "Bahamas sweep": [scaling["reference_sweep"]],
    "Four domains": scaling["domains"],
}
runs = [
    (domain, point)
    for domains in groups.values()
    for domain in domains
    for point in domain["points"]
]
tasks = np.array([point["ntasks_ocn"] for _, point in runs])
run_s = np.array([point["tot_run_time_s"] for _, point in runs])
a, b = fit(
    np.column_stack([np.ones_like(tasks), 1 / tasks]), run_s
)  # run time = a + b / N
sweep = sorted(
    scaling["reference_sweep"]["points"], key=lambda point: point["ntasks_ocn"]
)

checks.show(f"""
**The fit**, over {len(runs)} runs ({scaling['compset']}, {scaling['date_range'][0]} to {scaling['date_range'][1]}):

- run time (s) ≈ {a:.3g} + {b:.4g} / N, with N = NTASKS_OCN

{fit_error(a + b / tasks, run_s)}. Run time falls as 1/N toward a {a:.0f} s floor:
{sweep[0]['tot_run_time_s'] / sweep[-1]['tot_run_time_s']:.0f}× faster on {sweep[-1]['ntasks_ocn']} tasks than on {sweep[0]['ntasks_ocn']}.
""")

fig, ax = plt.subplots(figsize=(5, 3.6), layout="constrained")
span = np.geomspace(tasks.min(), tasks.max(), 40)
ax.loglog(span, a + b / span, "-", color="grey", label="fit")
for name, domains in groups.items():
    points = [point for domain in domains for point in domain["points"]]
    ax.loglog(
        [q["ntasks_ocn"] for q in points],
        [q["tot_run_time_s"] for q in points],
        "o",
        label=name,
    )
ax.set(title="Run time by tasks", xlabel="NTASKS_OCN (N)", ylabel="run time (s)")
ax.legend()
plain_numbers(fig)
plt.show()

In [ ]:
checks.show(
    checks.table(
        ["Domain", "NTASKS_OCN", "Simulated years/day", "Run time", "Fit"],
        [
            [
                domain["name"],
                point["ntasks_ocn"],
                f"{point['throughput_sim_years_per_day']:.1f}",
                fmt_s(point["tot_run_time_s"]),
                fmt_s(a + b / point["ntasks_ocn"]),
            ]
            for domain, point in runs
        ],
    )
)